In [7]:
import pandas as pd
import re
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
import sys
import torch


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [6]:
G_LLM = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
TEMP_PROCESSED_JSON = 'processed_data.json'
tokenizer = AutoTokenizer.from_pretrained(G_LLM)
model = AutoModelForCausalLM.from_pretrained(G_LLM).to("cuda")

Fetching 2 files:   0%|          | 0/2 [05:06<?, ?it/s]
Could not set the permissions on the file '/home/group1/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Qwen-7B/blobs/8b27be8d43b81d0cd19b3113ef88caf1a61c2fc602de77e3cce97aedcec184b4.incomplete'. Error: [Errno 2] No such file or directory: '/home/group1/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Qwen-7B/tmp_76b137f4-cca1-4d2e-bbed-63df9eb71058'.
Continuing without setting permissions.


KeyboardInterrupt: 

In [ ]:
pipeline = transformers.pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            device=0,
            temperature=0.1,
            do_sample=True,
            top_k=10,
            num_return_sequences=1,
            return_full_text=False,
            eos_token_id=tokenizer.eos_token_id
)


In [ ]:
risk_scores = []
def get_risk_score(source, header, content):
    """
    Generate risk score for each news/reddit/tweet entry using the provided pipeline.

    Args:
        pipeline (pipeline object): Text generation pipeline.
        source (str): Source of the data ("News", "Reddit", or "Tweet").
        header (str): Header/title of the news/reddit/tweet entry.
        content (str): Content/body of the news/reddit/tweet entry.

    Returns:
        int: Risk score assigned to the news/reddit/tweet entry.
    """
    try:
        if (source.lower() == "news") or (source.lower() == "reddit"):
            prompt_instruction = f"""### Instruction:
                    You are a financial expert specializing in risk assessment for Nvidia stock recommendations.
                    Given the headline and content of a {source.lower()} article, assess the risk level for investing in Nvidia stock.
                    """
        else:
            prompt_instruction = f"""### Instruction:
                    You are a financial expert specializing in risk assessment for Nvidia stock recommendations.
                    Given the content of a tweet, assess the stock market condition and how it relates to the
                    risk level for investing in Nvidia stock.
                    """

        prompt_body = f"""
                    Assign a risk score from 1 to 5, where:
                    1 = very low risk,
                    2 = low risk,
                    3 = moderate risk (use this if risk is unclear),
                    4 = high risk,
                    5 = very high risk.

                    ### Generation Rules:
                        - Only output an integer between 1 and 5 inclusive.
                        - Do not include any other text besides the number.
                        - The risk score should be determined by analyzing the sentiment expressed in the input text.
                        - Consider both positive and negative sentiments when assessing the risk level.
                        - Avoid assigning scores solely based on specific keywords without considering their context within the text.
                        - Think step-by-step before making your decision.

                    ### Input:
                    Headline: {header}
                    Content: {content}

                    ### Response:
                    Risk score:"""

        prompt = prompt_instruction + prompt_body

        response = pipeline(prompt)[0]['generated_text']

        # Extract risk score from response
        match = re.search(r'(\d)\s*<\/think>', response)
        if not match:
            match = re.search(r'\b([1-5])\b', response)

        if match:
            score = int(match.group(1))
            risk_scores.append(score)
            print(f"Extracted risk score: {score}")

        else:
            print("⚠️ Could not extract risk score from response:", response)
            risk_scores.append(3)

    except Exception as e:
        print(f"[Error in risk score generation] -> {e}")
        sys.exit(1)


def get_all_scores(json_data):
    """Function to get risk score for every news in the processed json file.

    Args:
        json_data (json object): The loaded data from reading processed_data.json.

    Returns:
        risk_score (list): A list of generated risk score for each news extracted.
    """
    for i in tqdm(range(min(25, len(json_data))), desc="Generating risk scores"):
        print(f"Data entry number {i}:\n")
        source = json_data[i]['source']
        header = json_data[i]['header']
        content = json_data[i]['content']

        score = get_risk_score(source, header, content)
        results = []
        results.append({
            "source": source,
            "header": header,
            "content": content,
            "risk_score": score
        })

        print(f"\nEntry {i}:")
        print(f"Header: {header}")
        print(f"Content: {content[:300]}...")

    print("Risk score generation completed.")

    return risk_scores

In [ ]:
import json
import sys

if __name__ == "__main__":

        # Get the processed data JSON file
        with open(TEMP_PROCESSED_JSON, 'r') as f:
            json_data = json.load(f)

        # Generate risk score for each news
        risk_scores = get_all_scores(json_data)
        print(risk_scores)

In [ ]:
import matplotlib.pyplot as plt

deepseek_1_5b = [3, 3, 3, 4, 4, 4, 3, 3, 3, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 5, 3, 3, 3, 4]
deepseek_7b   = [2, 2, 1, 4, 3, 2, 1, 4, 3, 3, 1, 1, 2, 4, 1, 3, 4, 1, 1, 2, 1, 2, 1, 1, 2]

risk_levels = ['1 -Very Low Risk', '2 -Low Risk', '3 -Moderate Risk (Neutral)', '4 -High Risk', '5 -Very High Risk']

matching_indices = [i for i, (a, b) in enumerate(zip(deepseek_1_5b, deepseek_7b)) if a == b]
num_matches = len(matching_indices)

plt.figure(figsize=(15, 4))
plt.plot(deepseek_1_5b, marker='o', label='DeepSeek 1.5B', linewidth=2)
plt.plot(deepseek_7b, marker='s', label='DeepSeek 7B', linewidth=2)
plt.axhline(y=3, color='red', linestyle='--', label='Neutral Risk (3)')

for idx in matching_indices:
    plt.scatter(idx, deepseek_1_5b[idx], s=150, facecolors='none', edgecolors='green', linewidths=2, label='Match' if idx == matching_indices[0] else "")

plt.title(f'Risk Score Predictions: DeepSeek 1.5B vs 7B (Matches: {num_matches}/25)')
plt.xlabel('Text Index')
plt.ylabel('Risk Level')
plt.xticks(range(len(deepseek_1_5b)), range(1, len(deepseek_1_5b)+1))
plt.yticks([1, 2, 3, 4, 5], risk_levels)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
